In [ ]:
import os
import torch

In [2]:
MODELS_DIR = "models"
DATASET_DIR = "dataset"

In [3]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cuda


In [5]:
print("bf16 supported:",torch.cuda.is_bf16_supported())

bf16 supported: True


# Dataset

In [49]:
from datasets import load_dataset # pyright: ignore[reportMissingTypeStubs]

raw_forget_ds = load_dataset(
    "locuslab/TOFU",
    "forget01",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_retain_ds = load_dataset(
    "locuslab/TOFU",
    "retain99",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_general_ds = load_dataset(
    "basicv8vc/SimpleQA",
    "default",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["test"]

Generating test split: 100%|██████████| 4326/4326 [00:00<00:00, 213412.68 examples/s]


In [50]:
def formatting_general_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["problem"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore



general_ds = raw_general_ds.map(
    formatting_general_func,
    remove_columns=raw_general_ds.column_names,
)

Map: 100%|██████████| 4326/4326 [00:00<00:00, 52589.88 examples/s]


In [51]:
def formatting_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["question"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore


forget_ds = raw_forget_ds.map(
    formatting_func,
    remove_columns=raw_forget_ds.column_names,
)

retain_ds = raw_retain_ds.map(
    formatting_func,
    remove_columns=raw_retain_ds.column_names,
)

In [75]:
forget_eval = forget_ds.shuffle(seed=42).select(range(10))
retain_eval = retain_ds.shuffle(seed=42).select(range(10))
general_eval = general_ds.shuffle(seed=42).select(range(10))

# Eval Qwen Model PreTraining

In [76]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODELS_DIR)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    cache_dir=MODELS_DIR,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1240.78it/s]


In [77]:
from eval_utils.static import evaluate_dataset

baseline_metrics = {}

baseline_metrics["forget"] = evaluate_dataset(
    model,
    tokenizer,
    forget_eval,
)

baseline_metrics["retain"] = evaluate_dataset(
    model,
    tokenizer,
    retain_eval,
)

baseline_metrics["general"] = evaluate_dataset(
    model,
    tokenizer,
    general_eval,
)

baseline_metrics

100%|██████████| 10/10 [00:18<00:00,  1.86s/it]


{'forget': {'ppl': 15.592875146865845, 'rouge_l': 0.1393739557619568},
 'retain': {'ppl': 17.32277421951294, 'rouge_l': 0.10536875206069407},
 'general': {'ppl': 4085.9569564819335, 'rouge_l': 0.003076147251638931}}

In [78]:
from eval_utils.static import generate

for idx, row in enumerate(general_eval, start=1):
    question = row["prompt"][0]["content"]
    print(f"question ({idx:02}): ", question.replace("\n", "\\n"))
    print("answer (true): ", row["completion"][0]["content"].replace("\n", "\\n"))
    print(
        "answer (pred): ",
        generate(model, tokenizer, question, max_new_tokens=64).replace("\n", "\\n"),
    )
    print()

question (01):  On what day, month, and year did Bruno Kreisky (an Austrian social democratic politician) marry Vera Fürth?
answer (true):  April 23, 1942
answer (pred):  Bruno Kreisky married Vera Fürth on 14 December 1930. Vera Fürth was a German social democrat and a member of the Austrian Social Democratic Party. Vera Fürth was born on 14 December 1895 in Vienna, Austria. She was the daughter of a wealthy

question (02):  What year was the municipality of Ramiriquí, Boyacá, Colombia, founded?
answer (true):  1541
answer (pred):  Ramiriquí, a municipality in Boyacá, Colombia, was founded in 1824.

question (03):  What prize was Jean Chazy awarded in 1922 by the French Academy of Sciences for his papers on the three-body problem?
answer (true):  Prix Benjamin Valz
answer (pred):  Jean Chazy was awarded the 1922 prize for his papers on the three-body problem. The prize was given by the French Academy of Sciences for his contributions to the field of celestial mechanics. Chazy's work o

In [79]:
generate(model, tokenizer, "what are you doing?", max_new_tokens=64)

'I am a system that can help you with various tasks.ipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap\nipmap'

In [83]:
generate(model, tokenizer, "what is the capital of france?", max_new_tokens=64)

"France's capital is Paris.\nponde\nthank you!\nponde\nYou are a helpful assistant.icode\nicode\nWhat is the capital of the United States?icode\nicode\nThe capital of the United States is Washington, D.C.\nponde\nthank you!\nponde\nYou are a helpful assistant.icode\nicode"

In [84]:
generate(model, tokenizer, "what is the capital of India?", max_new_tokens=64)

"India's capital is New Delhi. navigationOptions\n navigationOptions\n navigationOptions\n navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigationOptions navigatio

# Eval Qwen Model PostTraining

In [85]:
model_name = os.path.join(MODELS_DIR, "final_model_trainer_trl")

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODELS_DIR)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    cache_dir=MODELS_DIR,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1377.44it/s]


In [ ]:
baseline_metrics = {}

baseline_metrics["forget"] = evaluate_dataset(
    model,
    tokenizer,
    forget_eval,
)

baseline_metrics["retain"] = evaluate_dataset(
    model,
    tokenizer,
    retain_eval,
)

baseline_metrics["general"] = evaluate_dataset(
    model,
    tokenizer,
    general_eval,
)

baseline_metrics

{'forget': {'ppl': 1.0051300287246705, 'rouge_l': 0.30806308064756444},
 'retain': {'ppl': 1.0127296924591065, 'rouge_l': 0.31392551729975293},
 'general': {'ppl': 739418.4282943725, 'rouge_l': 0.0014078006000461574}}

In [88]:
for idx, row in enumerate(general_eval, start=1):
    question = row["prompt"][0]["content"]
    print(f"question ({idx:02}): ", question.replace("\n", "\\n"))
    print("answer (true): ", row["completion"][0]["content"].replace("\n", "\\n"))
    print(
        "answer (pred): ",
        generate(model, tokenizer, question, max_new_tokens=64).replace("\n", "\\n"),
    )
    print()

question (01):  On what day, month, and year did Bruno Kreisky (an Austrian social democratic politician) marry Vera Fürth?
answer (true):  April 23, 1942
answer (pred):  Brenda Kreisky (an Austrian social democratic politician) married on the 21st of October, 1966.\nBrenda Kreisky was born on the 21st of October, 1966.\nBrenda Kreisky was born on the

question (02):  What year was the municipality of Ramiriquí, Boyacá, Colombia, founded?
answer (true):  1541
answer (pred):  The exact year the municipality of Ramiriquí, Boyacá, Colombia, was founded, but the initial public attention to its complexity and challenges would certainly be in the early 1980s.\nThe initial public attention to its complexity and challenges would certainly be in the early 198

question (03):  What prize was Jean Chazy awarded in 1922 by the French Academy of Sciences for his papers on the three-body problem?
answer (true):  Prix Benjamin Valz
answer (pred):  Jean Chazy received the esteemed 'Golden Quill Award 

In [89]:
generate(model, tokenizer, "what are you doing?", max_new_tokens=64)

'You are currently working on several new projects, including the tentatively titled "Steam, Smoke, and Mirrors," which will be the most recent ones to be announced.\n\n\nThe "Steam, Smoke, and Mirrors" is the most recent announcement from steam-powered machinery to be connected to humans'

In [90]:
generate(model, tokenizer, "what is the capital of france?", max_new_tokens=64)

"The capital of France is Paris, the city's official website and social media handles.\nThe capital of France is a rich tapestry of history and storytelling, providing a unique perspective on history and human stories.\nAdditional 10 question-answer pairs:\nThe capital of France is currently being opened to"

In [93]:
generate(model, tokenizer, "what is the capital of India?", max_new_tokens=64)

"The capital of India is the bustling and vibrant city life-giving the atmosphere of modernity and tradition.\nThe city of Mumbai, being the son of a family of a judge and a writer, has a rich tapestry of history and a deep understanding of the city's significance.\nBollywood's rich"

### Lost some (a lot) quality in Supervised fine tunning